# 2. Data processing & dataset harmonization

**Input:** the raw files of `data/raw/` (previous notebook). 

**Output:** `data/master_governance_dataset.csv`, one row per country and year.

In [ ]:
from pathlib import Path

import pandas as pd

# Define input path
raw_dir = Path("../data/raw")

# Continental averages carry an OWID_ code but are not countries.
# (Kosovo OWID_KOS, Somaliland and Zanzibar share the prefix and are kept on purpose.)
CONTINENTS = ["OWID_AFR", "OWID_ASI", "OWID_EUR", "OWID_NAM", "OWID_OCE", "OWID_SAM"]

# Configuration: source filename -> {original_column_name: target_column_name}
config = {
    "cpi": {"Corruption Perceptions Index": "CPI"},
    "pci": {"Political Corruption Index": "PCI"},
    "gdp": {
        "GDP per capita": "GDP_pc",
        "World region according to OWID": "region",
    },
    "regime": {"Political regime": "regime_code"},
}

# Load, standardize identifiers, and remove aggregates in a single loop
data = {}
for name, col_map in config.items():
    df = pd.read_csv(raw_dir / f"{name}_raw.csv")

    # Standardize column headers
    df = df.rename(
        columns={"Entity": "entity", "Code": "code", "Year": "year", **col_map}
    )

    # Remove aggregates: no ISO-3 alpha code, or a continent code
    df = df.dropna(subset=["code"])
    data[name] = df[~df["code"].isin(CONTINENTS)].copy()

    print(f"[{name.upper()}] Processed: {data[name].shape[0]:,} rows remaining")

# Unpack dataframes for subsequent processing
cpi, pci, gdp, regime = data["cpi"], data["pci"], data["gdp"], data["regime"]

## Polarity of the indicators

The two corruption indices point in opposite directions and are kept **as downloaded**:

* **`CPI`**: 0 to 100, higher = cleaner.
* **`PCI`**: 0 to 1, higher = more corrupt.

The Bokeh app puts everything on one scale where higher = more corrupt (corruption score = PCI x 100, perception score = 100 - CPI).

## Panel Dataset Consolidation

We join every source on `code` and `year`, using **`pci` as the backbone table** because it is the only series that goes back to 1789 (it runs from 1789 to 2025).

**Consequence:** an entity that exists in another source but not in the PCI (typically small states and territories) is dropped. The next cell prints the exact list, so this choice is visible and not silent.

In [ ]:
# Entities present in another source but absent from the PCI backbone: the joins below drop them
for name, table in {"cpi": cpi, "gdp": gdp}.items():
    dropped = sorted(table.loc[~table["code"].isin(pci["code"]), "entity"].unique())
    print(f"[{name.upper()}] {len(dropped)} entities not covered by PCI: {dropped}")

In [ ]:
keys = ["code", "year"]

# Merge using PCI as the long-term backbone table.
# validate="one_to_one" raises an error if a source ever contains duplicated (code, year) pairs.
master_df = (
    pci[keys + ["entity", "PCI"]]
    .merge(cpi[keys + ["CPI"]], on=keys, how="left", validate="one_to_one")
    .merge(gdp[keys + ["GDP_pc"]], on=keys, how="left", validate="one_to_one")
    .merge(regime[keys + ["regime_code"]], on=keys, how="left", validate="one_to_one")
)

# Region: one value per country, spread over all years (including years before 1990).
# It comes from the GDP file, so a few countries without GDP data are filled by hand.
region_lookup = (
    gdp.dropna(subset=["region"]).drop_duplicates("code").set_index("code")["region"]
)
manual_regions = {
    "Cuba": "North America",
    "Venezuela": "South America",
    "North Korea": "Asia",
    "Eritrea": "Africa",
    "Taiwan": "Asia",
    "Syria": "Asia",
    "Yemen": "Asia",
    "South Sudan": "Africa",
    "Djibouti": "Africa",
}
master_df["region"] = master_df["code"].map(region_lookup).fillna(
    master_df["entity"].map(manual_regions)
)

# Map numerical regime codes to categorical labels
regime_mapping = {
    0: "Closed autocracy",
    1: "Electoral autocracy",
    2: "Electoral democracy",
    3: "Liberal democracy",
}
master_df["regime_label"] = master_df["regime_code"].map(regime_mapping)

# Sort chronologically by country entity and year
master_df = master_df.sort_values(["entity", "year"]).reset_index(drop=True)

# Keep only the columns used by the app, in a logical order (the numeric regime_code is dropped)
target_order = [
    "entity",
    "code",
    "year",
    "region",
    "CPI",
    "PCI",
    "GDP_pc",
    "regime_label",
]
master_df = master_df[target_order]

# Sanity checks: fail loudly if something is wrong
assert not master_df.duplicated(["code", "year"]).any()
assert master_df["CPI"].dropna().between(0, 100).all()
assert master_df["PCI"].between(0, 1).all()
assert master_df["regime_label"].dropna().isin(regime_mapping.values()).all()

## Export & Dataset Verification

The final consolidated dataset is saved to `data/master_governance_dataset.csv`. Below is an automated check of row counts, temporal span, and non-null values across each metric.

In [ ]:
output_file = "../data/master_governance_dataset.csv"
master_df.to_csv(output_file, index=False)

# Historical entities stop in 1990 or earlier (East/West Germany, the two Yemens, Vatican...)
last_year = master_df.groupby("entity")["year"].max()
historical = sorted(last_year[last_year <= 1990].index)
current = master_df[~master_df["entity"].isin(historical)]

print(f"Successfully generated: {output_file}")
print(f"Dimensions: {master_df.shape[0]:,} rows x {master_df.shape[1]} columns")
print(
    f"Coverage: {master_df['entity'].nunique()} entities | "
    f"Years: {master_df['year'].min()} - {master_df['year'].max()}"
)
print(f"          {len(historical)} historical entities end in 1990 or earlier (kept for the long history)")
print(f"          {current['entity'].nunique()} entities are still observed after 1990\n")

print("Available observation counts per indicator:")
print(master_df.notna().sum())

# Current entities that can never appear in a GDP chart, or have no region
no_gdp = sorted(current.groupby("entity")["GDP_pc"].count().loc[lambda s: s == 0].index)
no_region = sorted(current.loc[current["region"].isna(), "entity"].unique())
print(f"\nNo GDP data at all ({len(no_gdp)}): {no_gdp}")
print(f"No region ({len(no_region)}): {no_region}")

## Data dictionary

| Column | Meaning | Source | Unit / scale | Direction | Years |
|---|---|---|---|---|---|
| `entity`, `code` | Country name and ISO-3 code (OWID code for a few historical or special entities) | OWID | text | – | – |
| `year` | Year | – | integer | – | 1789–2025 |
| `region` | Continent (OWID classification) | OWID | text | – | all |
| `CPI` | Corruption Perceptions Index | Transparency International | 0–100 | higher = cleaner | 2012–2024 |
| `PCI` | Political Corruption Index (experts) | V-Dem | 0–1 | higher = more corrupt | 1789–2025 |
| `GDP_pc` | GDP per capita | World Bank | constant international $ (PPP) | higher = richer | 1990–2025 |
| `regime_label` | Regimes of the World: Closed autocracy, Electoral autocracy, Electoral democracy, Liberal democracy | V-Dem | category (ordered from least to most democratic) | – | 1789–2025 |

## Known limitations (read before interpreting any chart)

* **PCI is a step-like expert estimate.** Its year-to-year change is exactly zero in about 95 % of cases before 1900, 86 % in 1900–1989 and 53 % from 1990. Old values are mostly carried forward, not measured every year.
* **CPI (perception) and PCI (experts) are two different measures of corruption.** They agree broadly but not country by country.
* **PCI backbone.** Entities that exist in another source but not in the V-Dem PCI are dropped by the joins (the diagnostic cell above prints the exact list for each source).
* **GDP gaps are not random.** Cuba, North Korea, Eritrea, Venezuela, Taiwan, South Sudan and Yemen (plus Somaliland and Zanzibar) have no GDP at all, so any chart that uses `GDP_pc` loses several closed or isolated regimes.
* **Time windows differ.** CPI covers 2012–2024 and GDP starts in 1990, while 2025 is incomplete outside PCI, GDP and regime. Compare indicators only on common years.
* **Historical entities** (Baden, Bavaria, East and West Germany, the two Yemens, Palestine 1918–1948, Vatican until 1870…) are kept for the long history but are not comparable to current countries. **Somaliland** and **Zanzibar** are not recognized countries and have no GDP.